In [ ]:
#=============================================================================================================================
# CREDIT SCORING - PRÉTRAITEMENT DES DONNÉES
#==============================================================================================================================

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print ("✅ Librairies importées avec succès")
print (f"Pandas : {pd.__version__}")
print (f"Numpy : {pd.__version__}")

✅ Librairies importées avec succès
Pandas : 3.0.3
Numpy : 3.0.3


In [3]:
df = pd.read_csv('/Users/thierrymanuelzibiondobo/credit_scoring_project/data/raw/application_train.csv')

print(f"✅ Dataset chargé")
print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

✅ Dataset chargé
Dimensions : 307,511 lignes × 122 colonnes


In [4]:
df['DAYS_EMPLOYED_ANOMALY'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
print(df['DAYS_EMPLOYED_ANOMALY'].value_counts())

DAYS_EMPLOYED_ANOMALY
0    252137
1     55374
Name: count, dtype: int64


In [5]:
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

print(f"Valeurs manquantes DAYS_EMPLOYED : {df['DAYS_EMPLOYED'].isna().sum():,}")

Valeurs manquantes DAYS_EMPLOYED : 55,374


In [7]:
missing_pct = df.isnull().sum() / len(df)

cols_to_drop = missing_pct[missing_pct > 0.5].index.tolist()
print(f"Colonnes supprimées : {len(cols_to_drop)}")
print(cols_to_drop)

df = df.drop(columns=cols_to_drop)

Colonnes supprimées : 41
['OWN_CAR_AGE', 'EXT_SOURCE_1', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE']


In [8]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
numeric_cols = [ col for col in numeric_cols if col != 'TARGET']

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

print("✅Variables numériques imputées")

✅Variables numériques imputées


In [9]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("✅Variables catégorielles imputées")

✅Variables catégorielles imputées


In [10]:
print(f"Dimensions du dataset après suppression : {df.shape}")
print(f"\nValeurs manquantes restantes : {df.isnull().sum().sum()}")

Dimensions du dataset après suppression : (307511, 82)

Valeurs manquantes restantes : 0


In [11]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

binary_cols = [col for col in cat_cols if df[col].nunique()==2]
multi_cols = [col for col in cat_cols if df[col].nunique() > 2]

print(f"Variables binaires : {len(binary_cols)}")
print(f"Variables multi-modalités : {len(multi_cols)}")

Variables binaires : 4
Variables multi-modalités : 9


In [13]:
le = LabelEncoder()

for col in binary_cols:
    df[col] =  le.fit_transform(df[col].astype(str))

print("✅Variables binaires encodées")
print(df[binary_cols].head(3))

✅Variables binaires encodées
   NAME_CONTRACT_TYPE  FLAG_OWN_CAR  FLAG_OWN_REALTY  EMERGENCYSTATE_MODE
0                   0             0                1                    0
1                   0             0                0                    0
2                   1             1                1                    0


In [14]:
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

print("✅One-Hot Encoding éffectué")
print(f"Nouvelles dimensions : {df.shape}")

✅One-Hot Encoding éffectué
Nouvelles dimensions : (307511, 182)


In [15]:
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['TARGET'])
y = df['TARGET']

cols_to_scale = X.select_dtypes(include='number').columns.tolist()

scaler = StandardScaler()
X[cols_to_scale] = scaler.fit_transform(X[cols_to_scale])

print("✅Normalisation effectuée")
print(f"Features : {X.shape}")
print(f"Target : {y.shape}")

✅Normalisation effectuée
Features : (307511, 181)
Target : (307511,)


In [16]:
import os 

processed_dir = os.path.join(os.path.expanduser('~'),
                             'credit_scoring_project',
                             'data', 'processed')

X.to_csv(os.path.join(processed_dir, 'X_preprocessed.csv'), index=False)
y.to_csv(os.path.join(processed_dir, 'y_preprocessed.csv'), index=False)

print("✅Dataset preprocessé sauvegardé")
print(f"X : {X.shape}")
print(f"y : {y.shape}")

✅Dataset preprocessé sauvegardé
X : (307511, 181)
y : (307511,)
